# Rio Controller - Interactive Control Interface

This notebook provides an interactive UI for controlling the Rio microfluidics controller using ipywidgets.

## Features

- **Flow/Pressure Control**: Set flow rates and pressures with sliders
- **Heater Control**: Set temperatures and enable PID/stirrer
- **Camera**: Live stream and snapshot capture
- **Strobe**: Control strobe timing and enable/disable
- **Real-time Status**: Live updates of sensor readings
- **Emergency Stop**: Quick stop for all operations

## Prerequisites

1. Rio API server must be running (see `software/api/README.md`)
2. Install required packages:
   ```bash
   pip install requests websocket-client ipywidgets matplotlib pandas numpy
   ```


In [ ]:
# Configuration and imports
import os
import sys
from pathlib import Path
import json
import time
from datetime import datetime
from IPython.display import display, Image, HTML, clear_output
import ipywidgets as widgets
import threading

# Add software/client to path
repo_root = Path.cwd()
if (repo_root / "software" / "client").exists():
    sys.path.insert(0, str(repo_root / "software"))
elif (repo_root.parent.parent / "software" / "client").exists():
    sys.path.insert(0, str(repo_root.parent.parent))
else:
    for parent in repo_root.parents:
        if (parent / "software" / "client").exists():
            sys.path.insert(0, str(parent / "software"))
            break

from client import RioClient, RioStreamClient, RioAPIError

# API configuration
API_BASE_URL = os.getenv("RIO_API_URL", "http://localhost:8000")

# Initialize client
try:
    client = RioClient(base_url=API_BASE_URL)
    health = client.health()
    print(f"✅ Connected to Rio API at {API_BASE_URL}")
    print(f"   Status: {health['status']}, Simulation: {health['simulation']}")
except Exception as e:
    print(f"❌ Failed to connect to API: {e}")
    print(f"   Make sure the API server is running at {API_BASE_URL}")
    raise


## Flow/Pressure Control


In [ ]:
# Flow/Pressure Control UI
flow_channel = widgets.IntSlider(value=0, min=0, max=3, description="Channel:")
flow_rate = widgets.FloatSlider(value=0, min=0, max=1000, step=1, description="Flow (ul/hr):")
pressure_value = widgets.FloatSlider(value=0, min=0, max=200, step=1, description="Pressure (mbar):")
flow_status = widgets.Output(layout=widgets.Layout(height='150px', overflow_y='auto'))

def update_flow_state():
    """Update flow state display."""
    try:
        state = client.get_flow_state()
        with flow_status:
            clear_output()
            print("Current Flow/Pressure State:")
            for i in range(4):
                print(f"  Channel {i}:")
                print(f"    Flow: {state['flow_actuals_ul_hr'][i]:.1f} ul/hr (target: {state['flow_targets_ul_hr'][i]:.1f})")
                print(f"    Pressure: {state['pressure_actuals_mbar'][i]:.1f} mbar (target: {state['pressure_targets_mbar'][i]:.1f})")
                print(f"    Mode: {state['control_modes_text'][i]}")
    except Exception as e:
        with flow_status:
            clear_output()
            print(f"❌ Error: {e}")

def set_flow(change):
    """Set flow rate."""
    try:
        client.set_flow(flow_channel.value, flow_rate.value)
        update_flow_state()
    except Exception as e:
        with flow_status:
            print(f"❌ Error setting flow: {e}")

def set_pressure(change):
    """Set pressure."""
    try:
        client.set_pressure(flow_channel.value, pressure_value.value)
        update_flow_state()
    except Exception as e:
        with flow_status:
            print(f"❌ Error setting pressure: {e}")

flow_rate.observe(set_flow, 'value')
pressure_value.observe(set_pressure, 'value')

flow_controls = widgets.VBox([
    widgets.HTML("<h3>Flow/Pressure Control</h3>"),
    flow_channel,
    flow_rate,
    pressure_value,
    widgets.Button(description="🔄 Refresh Status", button_style='info'),
    flow_status
])

flow_controls.children[-2].on_click(lambda b: update_flow_state())
update_flow_state()

display(flow_controls)


## Heater Control


In [ ]:
# Heater Control UI
heater_index = widgets.IntSlider(value=0, min=0, max=3, description="Heater:")
heater_temp = widgets.FloatSlider(value=25, min=0, max=100, step=0.1, description="Temp (°C):")
heater_pid = widgets.ToggleButton(value=False, description="PID Control", button_style='info')
heater_stir = widgets.ToggleButton(value=False, description="Stirrer", button_style='info')
heater_status = widgets.Output(layout=widgets.Layout(height='150px', overflow_y='auto'))

def update_heater_state():
    """Update heater state display."""
    try:
        state = client.get_heater_state()
        with heater_status:
            clear_output()
            print("Current Heater States:")
            for i, h in enumerate(state['heaters']):
                print(f"  Heater {i}:")
                print(f"    Temp: {h['temp_c_actual']:.1f}°C (target: {h['temp_c_target']:.1f}°C)")
                print(f"    PID: {'ON' if h['pid_enabled'] else 'OFF'}")
                print(f"    Stir: {'ON' if h['stir_enabled'] else 'OFF'}")
                print(f"    Status: {h['status_text']}")
    except Exception as e:
        with heater_status:
            clear_output()
            print(f"❌ Error: {e}")

def set_heater_temp(change):
    """Set heater temperature."""
    try:
        client.set_heater_temp(heater_index.value, heater_temp.value)
        update_heater_state()
    except Exception as e:
        with heater_status:
            print(f"❌ Error: {e}")

def toggle_pid(change):
    """Toggle PID control."""
    try:
        client.set_heater_pid(heater_index.value, heater_pid.value)
        update_heater_state()
    except Exception as e:
        with heater_status:
            print(f"❌ Error: {e}")

def toggle_stir(change):
    """Toggle stirrer."""
    try:
        client.set_heater_stir(heater_index.value, heater_stir.value)
        update_heater_state()
    except Exception as e:
        with heater_status:
            print(f"❌ Error: {e}")

heater_temp.observe(set_heater_temp, 'value')
heater_pid.observe(toggle_pid, 'value')
heater_stir.observe(toggle_stir, 'value')

heater_controls = widgets.VBox([
    widgets.HTML("<h3>Heater Control</h3>"),
    heater_index,
    heater_temp,
    widgets.HBox([heater_pid, heater_stir]),
    widgets.Button(description="🔄 Refresh Status", button_style='info'),
    heater_status
])

heater_controls.children[-2].on_click(lambda b: update_heater_state())
update_heater_state()

display(heater_controls)


## Camera & Strobe


In [ ]:
# Camera and Strobe Control
camera_snapshot_btn = widgets.Button(description="📸 Capture Snapshot", button_style='success')
camera_display = widgets.Output(layout=widgets.Layout(width='400px', height='300px'))
strobe_enable = widgets.ToggleButton(value=False, description="Enable Strobe", button_style='success')
strobe_period = widgets.FloatSlider(value=50, min=1, max=10000, step=1, description="Period (µs):")
strobe_hold = widgets.ToggleButton(value=False, description="Hold Mode")

def capture_snapshot(btn):
    """Capture camera snapshot."""
    try:
        snapshot = client.get_camera_snapshot()
        with camera_display:
            clear_output()
            display(Image(data=snapshot, width=400))
    except Exception as e:
        with camera_display:
            clear_output()
            print(f"❌ Error: {e}")

def update_strobe(change):
    """Update strobe settings."""
    try:
        client.set_strobe_enable(strobe_enable.value)
        if strobe_period.value:
            client.set_strobe_timing(int(strobe_period.value * 1000))  # Convert to ns
        client.set_strobe_hold(strobe_hold.value)
    except Exception as e:
        print(f"❌ Error updating strobe: {e}")

camera_snapshot_btn.on_click(capture_snapshot)
strobe_enable.observe(update_strobe, 'value')
strobe_period.observe(update_strobe, 'value')
strobe_hold.observe(update_strobe, 'value')

camera_controls = widgets.VBox([
    widgets.HTML("<h3>Camera</h3>"),
    camera_snapshot_btn,
    camera_display,
    widgets.HTML("<h3>Strobe</h3>"),
    strobe_enable,
    strobe_period,
    strobe_hold
])

display(camera_controls)


## Emergency Stop


In [ ]:
# Emergency Stop
emergency_stop_btn = widgets.Button(
    description="🛑 EMERGENCY STOP",
    button_style='danger',
    layout=widgets.Layout(width='300px', height='50px')
)
emergency_status = widgets.Output()

def emergency_stop(btn):
    """Stop all operations immediately."""
    try:
        # Stop all flow channels
        for i in range(4):
            try:
                client.set_flow(i, 0.0)
                client.set_pressure(i, 0.0)
            except Exception:
                pass
        
        # Disable all heaters
        for i in range(4):
            try:
                client.set_heater_pid(i, False)
                client.set_heater_stir(i, False)
            except Exception:
                pass
        
        # Disable strobe
        try:
            client.set_strobe_enable(False)
            except Exception:
            pass
        
        with emergency_status:
            clear_output()
            print("🛑 EMERGENCY STOP: All operations stopped")
            print("   - All flow channels set to 0")
            print("   - All heaters disabled")
            print("   - Strobe disabled")
    except Exception as e:
        with emergency_status:
            print(f"❌ Emergency stop error: {e}")

emergency_stop_btn.on_click(emergency_stop)

display(widgets.VBox([
    emergency_stop_btn,
    emergency_status
]))
